# 87 — Campaign B (S2) autonomous exploration

Campaign C (S3) が screening q>=1 のまま尽きたあとの **Campaign B** 無人探索。

- `src.campaign_b.driver.run_campaign_b` のみ起動
- staged j2>=2 のみ（j2=1 / notebook 73 に送らない）
- 共有 M2 が無ければ **自動で staged j2=2 canonical を生成・registry 登録**（84/73 相当）
- production M6・Campaign C 再発行・証明書化は禁止
- 全成果物 `NOT_CERTIFIED` / `SCREENING_ONLY`
- `never_stop` + soft wall-clock。止まっても永続ポインタから自動再開

再開 ID の優先順位:

1. 環境変数 `VALIDATED_RG_M7B_RESUME_ID`
2. 永続ファイル `$PERSIST/campaign_b/LATEST_CAMPAIGN_B_RESUME.json`
3. 無ければ新規 run


In [ ]:
from pathlib import Path
import os, sys, json, shutil

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'campaign_b' / 'driver.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/campaign_b/driver.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)


In [ ]:
from src.runtime import validate_persistent_root
from src.campaign_b.resume_pointer import (
    RESUME_ENV_KEY,
    read_resume_id,
    resume_pointer_path,
)

validate_persistent_root()

BASE_CONFIG = PROJECT_ROOT / 'configs' / 'campaign_b_six_hour.yaml'
SPACE_CONFIG = PROJECT_ROOT / 'configs' / 'campaign_b_s2_space_v1.yaml'
assert BASE_CONFIG.is_file(), BASE_CONFIG
assert SPACE_CONFIG.is_file(), SPACE_CONFIG

RUNTIME_DIR = PERSIST_ROOT / 'campaign_b' / '_runtime'
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_SPACE = RUNTIME_DIR / 'campaign_b_s2_space_v1.yaml'
RUNTIME_CONFIG = RUNTIME_DIR / 'campaign_b_run.yaml'
shutil.copy2(SPACE_CONFIG, RUNTIME_SPACE)

RESUME_ID = (os.environ.get(RESUME_ENV_KEY) or '').strip() or None
RESUME_SOURCE = 'env' if RESUME_ID else None
if not RESUME_ID:
    RESUME_ID = read_resume_id(PERSIST_ROOT)
    RESUME_SOURCE = 'persist_pointer' if RESUME_ID else None
if RESUME_ID:
    os.environ[RESUME_ENV_KEY] = RESUME_ID

lines = BASE_CONFIG.read_text(encoding='utf-8').splitlines()
out = []
for line in lines:
    if line.startswith('persistent_root:'):
        out.append(f'persistent_root: {PERSIST_ROOT}')
    elif line.startswith('search_space_path:'):
        out.append(f'search_space_path: {RUNTIME_SPACE.name}')
    else:
        out.append(line)
if RESUME_ID:
    out.append(f'resume_campaign_run_id: {RESUME_ID}')
    out.append('inherit_deadline: false')
RUNTIME_CONFIG.write_text('\n'.join(out) + '\n', encoding='utf-8')
print('RUNTIME_CONFIG', RUNTIME_CONFIG)
print('RESUME_ID', RESUME_ID, 'source', RESUME_SOURCE)
print('RESUME_POINTER', resume_pointer_path(PERSIST_ROOT))


In [ ]:
from src.campaign_b import run_campaign_b
from src.campaign_b.resume_pointer import read_resume_id, resume_pointer_path

SUMMARY = run_campaign_b(RUNTIME_CONFIG)
print(json.dumps({
    'terminal_reason': SUMMARY.get('terminal_reason'),
    'campaign_run_id': SUMMARY.get('campaign_run_id'),
    'selected_count': SUMMARY.get('selected_count'),
    'archived_count': SUMMARY.get('archived_count'),
    'pending_count': SUMMARY.get('pending_count'),
    'certification_status': SUMMARY.get('certification_status'),
    'claim_scope': SUMMARY.get('claim_scope'),
    'restart_hint': SUMMARY.get('restart_hint'),
    'resume_pointer': SUMMARY.get('resume_pointer'),
}, indent=2, ensure_ascii=False, default=str))

persisted = read_resume_id(PERSIST_ROOT)
print('persisted VALIDATED_RG_M7B_RESUME_ID =', persisted)
print('pointer file', resume_pointer_path(PERSIST_ROOT))
print('export script', PERSIST_ROOT / 'campaign_b' / 'export_VALIDATED_RG_M7B_RESUME_ID.sh')


## 終了理由の見方

| terminal_reason | 意味 |
|---|---|
| `B_Q_LT_1_LINEAGE_READY` | screening q&lt;1 + 独立検証まで到達（**証明書ではない**） |
| `B_SCREENING_EXHAUSTED` | 許可空間を処理し尽くした |
| `B_BLOCKED_NEED_CANONICAL_M2` | 共有 M2 が無い → notebook **84** / **73 staged** で canonical を用意 |
| `B_FAIL_CLOSED` | audit / invariant 失敗。成果物を証明書扱いしない |
| `B_TIME_BUDGET_EXHAUSTED` | `enforce_wall_clock: true` のときのみ |

再開は notebook を再実行するだけでよい（ポインタを自動読込）。手動なら:

```bash
source /storage/validated_4d_su2_rg/campaign_b/export_VALIDATED_RG_M7B_RESUME_ID.sh
```
